# Barra-Like Return Attribution for CMM

This notebook decomposes the CMM long-short portfolio return using a simplified Barra-style cross-sectional factor model.

For each signal month, estimate stock next-month returns on:

- industry dummies (`ind1`),
- style exposures: Size, Liquidity, Momentum, Volatility, Profitability, Growth, Leverage.

Then decompose the realized CMM portfolio return into:

```text
portfolio return = market/intercept contribution + style contribution + industry contribution + specific return
```

The portfolio weights are generated with the same monthly rebalance, limit-up buy constraint, and limit-down sell constraint used in the main backtest.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DATA_PATH = Path('../output/datasets/cmm_model_training_data.parquet')
PRED_PATH = Path('../output/models/cmm/cmm_predictions.parquet')
RETURN_COLS_PATH = Path('../output/datasets/cmm_return_window_columns.txt')
DAILY_DIR = Path('../../data/daily')
OUT_DIR = Path('../output/reports/barra_attribution')
OUT_DIR.mkdir(parents=True, exist_ok=True)

EVAL_SPLIT = 'test'
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)


## 1. Load Test Sample, CMM Signal, and Exposures

In [ ]:
return_cols = RETURN_COLS_PATH.read_text(encoding='utf-8').splitlines()

style_raw_cols = [
    'z_tmv', 'z_nmv', 'z_amount', 'z_volume', 'z_listing_days',
    'z_roe_ttm', 'z_roa_ttm', 'z_net_profit_ratio_ttm', 'z_gross_profit_ratio_ttm',
    'z_oper_revenue_gr_ttm', 'z_net_profit_gr_ttm', 'z_eps_gr_ttm', 'z_net_asset_gr',
    'z_debt_assets_ratio', 'z_debt_to_equity_ratio', 'z_equity_multiplier',
]

base_cols = list(dict.fromkeys([
    'stock_id', 'signal_date', 'trade_date', 'target_1m_ret', 'ind1', 'z_tmv'
] + style_raw_cols + return_cols))
base = pd.read_parquet(DATA_PATH, columns=base_cols)
base['signal_date'] = pd.to_datetime(base['signal_date'])
base['trade_date'] = pd.to_datetime(base['trade_date'])
base['ind1'] = base['ind1'].fillna('Unknown')

pred = pd.read_parquet(PRED_PATH)
pred['signal_date'] = pd.to_datetime(pred['signal_date'])
pred = pred[pred['split'] == EVAL_SPLIT].copy()

df = pred[['stock_id', 'signal_date', 'trade_date', 'target_1m_ret', 'cmm_signal_cs_z']].merge(
    base, on=['stock_id', 'signal_date', 'trade_date', 'target_1m_ret'], how='inner'
)

# Standard momentum exposure using the paper window.
df['style_momentum'] = df[return_cols].sum(axis=1)
# Realized return volatility in the formation window.
df['style_volatility'] = df[return_cols].std(axis=1, ddof=0)

def cs_z(series):
    s = pd.to_numeric(series, errors='coerce').astype(float)
    std = s.std(ddof=0)
    if not np.isfinite(std) or std == 0:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / std

# Barra-like style composites. All are re-standardized by month.
df['style_size'] = df['z_tmv']
df['style_liquidity'] = df[['z_amount', 'z_volume']].mean(axis=1)
df['style_profitability'] = df[[c for c in ['z_roe_ttm', 'z_roa_ttm', 'z_net_profit_ratio_ttm', 'z_gross_profit_ratio_ttm'] if c in df.columns]].mean(axis=1)
df['style_growth'] = df[[c for c in ['z_oper_revenue_gr_ttm', 'z_net_profit_gr_ttm', 'z_eps_gr_ttm', 'z_net_asset_gr'] if c in df.columns]].mean(axis=1)
df['style_leverage'] = df[[c for c in ['z_debt_assets_ratio', 'z_debt_to_equity_ratio', 'z_equity_multiplier'] if c in df.columns]].mean(axis=1)

style_cols = ['style_size', 'style_liquidity', 'style_momentum', 'style_volatility', 'style_profitability', 'style_growth', 'style_leverage']
for col in style_cols:
    df[col] = df.groupby('signal_date')[col].transform(cs_z).fillna(0.0)

print(df.shape)
print(df['signal_date'].min(), df['signal_date'].max(), df['signal_date'].nunique())
df[['stock_id', 'signal_date', 'cmm_signal_cs_z', 'target_1m_ret', 'ind1'] + style_cols].head()


## 2. Load Trading Constraints and Market Caps

In [ ]:
def normalize_daily_symbol(symbol: pd.Series) -> pd.Series:
    s = symbol.astype(str).str.strip()
    exchange = s.str[:2]
    code = s.str[2:]
    suffix = exchange.map({'SH': '.SH', 'SZ': '.SZ', 'BJ': '.BJ'}).fillna('')
    return code + suffix


def limit_rate(stock_id: pd.Series, trade_date: pd.Series, st: pd.Series | None = None) -> pd.Series:
    code = stock_id.astype(str).str[:6]
    date = pd.to_datetime(trade_date)
    rate = pd.Series(0.10, index=stock_id.index, dtype=float)
    rate.loc[code.str.startswith('688') & (date >= pd.Timestamp('2019-07-22'))] = 0.20
    rate.loc[code.str.startswith('300') & (date >= pd.Timestamp('2020-08-24'))] = 0.20
    rate.loc[code.str.startswith(('8', '4'))] = 0.30
    if st is not None:
        rate.loc[pd.to_numeric(st, errors='coerce').fillna(0).astype(int).eq(1)] = 0.05
    return rate


def load_trade_flags(trade_dates: pd.Series) -> pd.DataFrame:
    pieces = []
    for d in sorted(pd.to_datetime(trade_dates.dropna().unique())):
        daily = pd.read_csv(DAILY_DIR / f'{d:%Y-%m-%d}.csv', usecols=['date', 'symbol', 'close', 'preClose', 'ret', 'st'])
        daily['trade_date'] = pd.to_datetime(daily['date'])
        daily['stock_id'] = normalize_daily_symbol(daily['symbol'])
        rate = limit_rate(daily['stock_id'], daily['trade_date'], daily['st'])
        ret = pd.to_numeric(daily['ret'], errors='coerce')
        close = pd.to_numeric(daily['close'], errors='coerce')
        pre_close = pd.to_numeric(daily['preClose'], errors='coerce')
        daily['limit_up'] = (ret >= rate - 0.002) | (close >= pre_close * (1 + rate) * 0.998)
        daily['limit_down'] = (ret <= -rate + 0.002) | (close <= pre_close * (1 - rate) * 1.002)
        pieces.append(daily[['stock_id', 'trade_date', 'limit_up', 'limit_down']])
    return pd.concat(pieces, ignore_index=True).drop_duplicates(['stock_id', 'trade_date'])


def load_signal_market_caps(signal_dates: pd.Series) -> pd.DataFrame:
    pieces = []
    for d in sorted(pd.to_datetime(signal_dates.dropna().unique())):
        daily = pd.read_csv(DAILY_DIR / f'{d:%Y-%m-%d}.csv', usecols=['date', 'symbol', 'tmv'])
        daily['signal_date'] = pd.to_datetime(daily['date'])
        daily['stock_id'] = normalize_daily_symbol(daily['symbol'])
        daily['market_cap'] = pd.to_numeric(daily['tmv'], errors='coerce')
        pieces.append(daily[['stock_id', 'signal_date', 'market_cap']])
    return pd.concat(pieces, ignore_index=True).drop_duplicates(['stock_id', 'signal_date'])

df = df.merge(load_signal_market_caps(df['signal_date']), on=['stock_id', 'signal_date'], how='left')
df = df.merge(load_trade_flags(df['trade_date']), on=['stock_id', 'trade_date'], how='left')
df[['limit_up', 'limit_down']] = df[['limit_up', 'limit_down']].fillna(False).astype(bool)
print('market cap coverage:', df['market_cap'].notna().mean())
print('limit flags:', df[['limit_up', 'limit_down']].mean().to_dict())


## 3. Reconstruct CMM Long-Short Portfolio Weights

Weights are reconstructed separately for equal-weighted and value-weighted versions, with limit-up buy and limit-down sell constraints. The combined long-short weight is long decile 10 minus short decile 1.

In [ ]:
def assign_deciles(frame: pd.DataFrame, signal_col: str) -> pd.DataFrame:
    pieces = []
    for date, group in frame.groupby('signal_date', sort=True):
        group = group.dropna(subset=[signal_col, 'target_1m_ret']).copy()
        if len(group) < 100 or group[signal_col].nunique() < 10:
            continue
        group['decile'] = pd.qcut(group[signal_col].rank(method='first'), 10, labels=False) + 1
        pieces.append(group)
    return pd.concat(pieces, ignore_index=True)


def target_weights_for_decile(month: pd.DataFrame, decile: int, weighting: str) -> dict[str, float]:
    target = month.loc[month['decile'] == decile, ['stock_id', 'market_cap']].copy()
    if target.empty:
        return {}
    if weighting == 'equal':
        return {name: 1.0 / len(target) for name in target['stock_id']}
    cap = pd.to_numeric(target['market_cap'], errors='coerce').clip(lower=0).fillna(0.0)
    if cap.sum() <= 0:
        return {name: 1.0 / len(target) for name in target['stock_id']}
    return dict(zip(target['stock_id'], cap / cap.sum()))


def rebalance(weights, cash, target_weight, can_buy, can_sell):
    target_names = set(target_weight)
    new_weights = dict(weights)
    for name in list(new_weights):
        current = new_weights.get(name, 0.0)
        target = target_weight.get(name, 0.0)
        if current > target and can_sell.get(name, True):
            cash += current - target
            if target > 1e-12:
                new_weights[name] = target
            else:
                new_weights.pop(name, None)
    buy_gaps = {}
    for name in target_names:
        current = new_weights.get(name, 0.0)
        target = target_weight.get(name, 0.0)
        if target > current and can_buy.get(name, True):
            buy_gaps[name] = target - current
    total_gap = sum(buy_gaps.values())
    if total_gap > 0 and cash > 0:
        scale = min(1.0, cash / total_gap)
        spent = 0.0
        for name, gap in buy_gaps.items():
            add = gap * scale
            new_weights[name] = new_weights.get(name, 0.0) + add
            spent += add
        cash -= spent
    return {k: v for k, v in new_weights.items() if v > 1e-12}, max(cash, 0.0)


def build_ls_weights(frame: pd.DataFrame, signal_col: str, weighting: str) -> tuple[pd.DataFrame, pd.Series]:
    assigned = assign_deciles(frame, signal_col)
    long_state = {'weights': {}, 'cash': 1.0}
    short_state = {'weights': {}, 'cash': 1.0}
    weight_rows = []
    ls_returns = []

    for date in sorted(assigned['signal_date'].unique()):
        month = assigned[assigned['signal_date'] == date].copy()
        returns = month.set_index('stock_id')['target_1m_ret'].to_dict()
        can_buy = (~month.set_index('stock_id')['limit_up']).to_dict()
        can_sell = (~month.set_index('stock_id')['limit_down']).to_dict()

        long_target = target_weights_for_decile(month, 10, weighting)
        short_target = target_weights_for_decile(month, 1, weighting)
        long_w, long_cash = rebalance(long_state['weights'], long_state['cash'], long_target, can_buy, can_sell)
        short_w, short_cash = rebalance(short_state['weights'], short_state['cash'], short_target, can_buy, can_sell)

        long_ret = sum(w * float(returns.get(name, 0.0)) for name, w in long_w.items())
        short_ret = sum(w * float(returns.get(name, 0.0)) for name, w in short_w.items())
        ls_returns.append({'signal_date': date, 'portfolio_return': long_ret - short_ret})

        for name, w in long_w.items():
            weight_rows.append({'signal_date': date, 'stock_id': name, 'portfolio_weight': w, 'side': 'long'})
        for name, w in short_w.items():
            weight_rows.append({'signal_date': date, 'stock_id': name, 'portfolio_weight': -w, 'side': 'short'})

        def drift(weights, cash):
            end_value = cash + sum(w * (1.0 + float(returns.get(name, 0.0))) for name, w in weights.items())
            if end_value <= 0:
                return {}, 0.0
            next_w = {name: w * (1.0 + float(returns.get(name, 0.0))) / end_value for name, w in weights.items()}
            next_cash = cash / end_value
            return next_w, next_cash

        long_state['weights'], long_state['cash'] = drift(long_w, long_cash)
        short_state['weights'], short_state['cash'] = drift(short_w, short_cash)

    weights = pd.DataFrame(weight_rows)
    ls_ret = pd.DataFrame(ls_returns).set_index('signal_date')['portfolio_return'].sort_index()
    return weights, ls_ret

weights_ew, ls_ew = build_ls_weights(df, 'cmm_signal_cs_z', 'equal')
weights_vw, ls_vw = build_ls_weights(df, 'cmm_signal_cs_z', 'value')
print('EW weights', weights_ew.shape, 'months', ls_ew.shape)
print('VW weights', weights_vw.shape, 'months', ls_vw.shape)
print('EW ann ret', ls_ew.mean() * 12, 'VW ann ret', ls_vw.mean() * 12)


## 4. Estimate Monthly Barra-Like Factor Returns

In [ ]:
def monthly_factor_model(frame: pd.DataFrame, style_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    factor_returns = []
    residual_rows = []

    for date, g in frame.groupby('signal_date', sort=True):
        g = g.dropna(subset=['target_1m_ret']).copy()
        if len(g) < 200:
            continue
        y = g['target_1m_ret'].to_numpy(dtype=float)
        styles = g[style_cols].astype(float).reset_index(drop=True)
        industry = pd.get_dummies(g['ind1'].fillna('Unknown').astype(str).reset_index(drop=True), prefix='ind', drop_first=True, dtype=float)
        x = pd.concat([pd.DataFrame({'Intercept': np.ones(len(g))}), styles.reset_index(drop=True), industry], axis=1)
        beta, *_ = np.linalg.lstsq(x.to_numpy(dtype=float), y, rcond=None)
        fitted = x.to_numpy(dtype=float) @ beta
        resid = y - fitted

        row = {'signal_date': date}
        row.update(dict(zip(x.columns, beta)))
        factor_returns.append(row)

        rr = g[['stock_id', 'signal_date']].copy()
        rr['specific_return'] = resid
        rr['fitted_return'] = fitted
        residual_rows.append(rr)

    return pd.DataFrame(factor_returns).set_index('signal_date').sort_index(), pd.concat(residual_rows, ignore_index=True)

factor_returns, residuals = monthly_factor_model(df, style_cols)
factor_returns = factor_returns.fillna(0.0)
industry_cols = [c for c in factor_returns.columns if c.startswith('ind_')]
print(factor_returns.shape, residuals.shape)
factor_returns[style_cols].describe().T


## 5. Attribute CMM Portfolio Returns

In [ ]:
def attribute_portfolio(weights: pd.DataFrame, realized_returns: pd.Series, label: str) -> pd.DataFrame:
    exposures = df[['stock_id', 'signal_date', 'target_1m_ret', 'ind1'] + style_cols].merge(
        residuals, on=['stock_id', 'signal_date'], how='left'
    )
    w = weights.merge(exposures, on=['stock_id', 'signal_date'], how='left')
    rows = []

    for date, g in w.groupby('signal_date', sort=True):
        if date not in factor_returns.index:
            continue
        fr = factor_returns.loc[date]
        row = {'signal_date': date, 'portfolio': label}
        row['realized_return'] = realized_returns.loc[date]
        row['direct_weighted_return'] = (g['portfolio_weight'] * g['target_1m_ret']).sum()
        row['intercept_contribution'] = g['portfolio_weight'].sum() * fr['Intercept']

        style_total = 0.0
        for col in style_cols:
            exposure = (g['portfolio_weight'] * g[col]).sum()
            contrib = exposure * fr[col]
            row[f'exposure_{col}'] = exposure
            row[f'contrib_{col}'] = contrib
            style_total += contrib
        row['style_contribution'] = style_total

        # Industry contribution uses the same drop-first encoding: omitted industry is absorbed into intercept.
        industry_total = 0.0
        for col in industry_cols:
            ind_name = col.replace('ind_', '', 1)
            exposure = g.loc[g['ind1'].astype(str).eq(ind_name), 'portfolio_weight'].sum()
            contrib = exposure * fr[col]
            row[f'exposure_{col}'] = exposure
            row[f'contrib_{col}'] = contrib
            industry_total += contrib
        row['industry_contribution'] = industry_total

        row['specific_contribution'] = (g['portfolio_weight'] * g['specific_return']).sum()
        row['explained_contribution'] = row['intercept_contribution'] + row['style_contribution'] + row['industry_contribution']
        row['reconstructed_return'] = row['explained_contribution'] + row['specific_contribution']
        row['reconstruction_error'] = row['direct_weighted_return'] - row['reconstructed_return']
        rows.append(row)

    return pd.DataFrame(rows).sort_values('signal_date')

attr_ew = attribute_portfolio(weights_ew, ls_ew, 'CMM_equal_weight')
attr_vw = attribute_portfolio(weights_vw, ls_vw, 'CMM_value_weight')
attr = pd.concat([attr_ew, attr_vw], ignore_index=True)
attr.to_csv(OUT_DIR / 'cmm_barra_monthly_attribution.csv', index=False)
print(attr.groupby('portfolio')[['realized_return', 'explained_contribution', 'style_contribution', 'industry_contribution', 'specific_contribution', 'reconstruction_error']].mean() * 12)
print('max abs reconstruction error', attr['reconstruction_error'].abs().max())


## 6. Contribution Summary and Charts

In [ ]:
def perf(series):
    s = series.dropna()
    ann = s.mean() * 12
    vol = s.std(ddof=1) * np.sqrt(12)
    return pd.Series({
        'annualized_mean': ann,
        'annualized_vol': vol,
        'sharpe_like': ann / vol if vol > 0 else np.nan,
        'monthly_t_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 1 and s.std(ddof=1) > 0 else np.nan,
    })

summary_parts = []
for portfolio, g in attr.groupby('portfolio'):
    cols = ['realized_return', 'intercept_contribution', 'style_contribution', 'industry_contribution', 'specific_contribution', 'explained_contribution']
    tmp = g[cols].apply(perf).T.reset_index().rename(columns={'index': 'component'})
    tmp.insert(0, 'portfolio', portfolio)
    summary_parts.append(tmp)
summary = pd.concat(summary_parts, ignore_index=True)
summary.to_csv(OUT_DIR / 'cmm_barra_attribution_summary.csv', index=False)
summary


In [ ]:
# Style contribution breakdown.
style_contrib_cols = [f'contrib_{c}' for c in style_cols]
style_rows = []
for portfolio, g in attr.groupby('portfolio'):
    for raw_col, style in zip(style_contrib_cols, style_cols):
        s = g[raw_col].dropna()
        style_rows.append({
            'portfolio': portfolio,
            'style': style.replace('style_', ''),
            'annualized_contribution': s.mean() * 12,
            'monthly_t_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 1 and s.std(ddof=1) > 0 else np.nan,
            'positive_rate': (s > 0).mean(),
        })
style_summary = pd.DataFrame(style_rows)
style_summary.to_csv(OUT_DIR / 'cmm_style_contribution_summary.csv', index=False)
style_summary


In [ ]:
# Industry contribution breakdown: top absolute annualized contributions.
industry_contrib_cols = [f'contrib_{c}' for c in industry_cols]
industry_rows = []
for portfolio, g in attr.groupby('portfolio'):
    for col in industry_contrib_cols:
        s = g[col].dropna()
        industry_rows.append({
            'portfolio': portfolio,
            'industry': col.replace('contrib_ind_', '', 1),
            'annualized_contribution': s.mean() * 12,
            'monthly_t_stat': s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 1 and s.std(ddof=1) > 0 else np.nan,
        })
industry_summary = pd.DataFrame(industry_rows)
industry_summary['abs_contribution'] = industry_summary['annualized_contribution'].abs()
industry_summary = industry_summary.sort_values(['portfolio', 'abs_contribution'], ascending=[True, False])
industry_summary.to_csv(OUT_DIR / 'cmm_industry_contribution_summary.csv', index=False)
industry_summary.groupby('portfolio').head(10)


In [ ]:
# Plot cumulative contribution by component.
component_cols = ['intercept_contribution', 'style_contribution', 'industry_contribution', 'specific_contribution']
plot_paths = []
for portfolio, g in attr.groupby('portfolio'):
    g = g.sort_values('signal_date').set_index('signal_date')
    cumulative = g[component_cols].cumsum()
    fig, ax = plt.subplots(figsize=(10, 5))
    cumulative.plot(ax=ax, linewidth=2)
    ax.set_title(f'Cumulative Barra-Like Return Attribution: {portfolio}')
    ax.set_xlabel('Signal date')
    ax.set_ylabel('Cumulative arithmetic contribution')
    fig.tight_layout()
    path = OUT_DIR / f'cumulative_attribution_{portfolio}.png'
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    plot_paths.append(path)
plot_paths


In [ ]:
# Plot annualized style contributions.
plot_paths = []
for portfolio, g in style_summary.groupby('portfolio'):
    gg = g.sort_values('annualized_contribution')
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.barh(gg['style'], gg['annualized_contribution'])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Annualized Style Contribution: {portfolio}')
    ax.set_xlabel('Annualized contribution')
    fig.tight_layout()
    path = OUT_DIR / f'style_contribution_{portfolio}.png'
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)
    plot_paths.append(path)
plot_paths
